# ReportLens — Explainer LoRA fine-tuning (Colab)

QLoRA fine-tunes a small open instruct model (Qwen2.5-3B) on the ReportLens explanation
style + safety rules, then exports **GGUF** so **Ollama** can serve it — no external AI
API, which is the whole point of ReportLens.

Just run the cells top-to-bottom. First set **Runtime → Change runtime type → T4 GPU**.

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Clone the repo (always pulls the latest code)

If the repo is already there from an earlier run, it is **reset to the latest `main`** —
otherwise a stale checkout would silently keep running old code. The cell prints the
commit it's running so you can confirm you have the newest fixes.

In [ ]:
%cd /content
# Clone if missing, otherwise hard-reset to the latest main. Re-running the notebook in an
# existing session must NOT keep a stale checkout (that silently hides pushed fixes).
# This clone is a disposable training checkout - don't edit files here.
!if [ -d reportlens ]; then \
   cd reportlens && git fetch origin && git reset --hard origin/main; \
 else \
   git clone https://github.com/Satwiksingh123/reportlens-backend-.git reportlens; \
 fi
%cd /content/reportlens
!echo "--- running commit ---" && git log --oneline -1

## 3. Install deps (Unsloth + friends, ~3-4 min)
`numpy` is needed by the RAG teacher used to build the dataset.

In [ ]:
!pip install -q -e "services/llm_service[finetune]"
!pip install -q numpy

## 4. Build the training dataset
Generates 300 synthetic reports, runs them through the parser + RAG-grounded teacher, and
writes `explainer_sft.jsonl` (hand-written seed examples are included automatically).

In [ ]:
%cd /content/reportlens/services/llm_service
!python -m llm_service.finetune.build_dataset --num-reports 300 \
    --out artifacts/explainer_sft.jsonl
!head -n 1 artifacts/explainer_sft.jsonl

## 5. Fine-tune (QLoRA) and export GGUF for Ollama
~15-25 min on a T4. Saves the LoRA adapter and a q4_k_m GGUF.

In [ ]:
!python -m llm_service.finetune.train_lora \
    --data artifacts/explainer_sft.jsonl \
    --base unsloth/Qwen2.5-3B-Instruct-bnb-4bit \
    --out artifacts/explainer-lora --epochs 2 --export-gguf

## 6. Quick generation test with the fine-tuned adapter

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from llm_service.prompts import SYSTEM_PROMPT, BiomarkerContext, build_biomarker_prompt

model, tok = FastLanguageModel.from_pretrained('artifacts/explainer-lora', max_seq_length=2048, load_in_4bit=True)
tok = get_chat_template(tok, chat_template='qwen-2.5')
FastLanguageModel.for_inference(model)

ctx = BiomarkerContext(test_name='Ferritin', value='8', unit='ng/mL', reference_range='30-300', status='Low', reference_notes='[MedlinePlus] Ferritin reflects stored iron; low ferritin is an early sign of iron deficiency.')
msgs = [{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':build_biomarker_prompt(ctx)}]
inputs = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=180, temperature=0.3)
print(tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## 7. Download the GGUF (to serve with Ollama on your machine)
The model is too big for git — keep it as a release asset / on Drive.

In [ ]:
!cd artifacts && zip -qr explainer-lora-gguf.zip explainer-lora-gguf
from google.colab import files
files.download('artifacts/explainer-lora-gguf.zip')

### Serve it with Ollama (on your own machine — self-hosted, free)
```bash
unzip explainer-lora-gguf.zip
ollama create reportlens-explainer -f explainer-lora-gguf/Modelfile
# then in the API .env:  OLLAMA_MODEL=reportlens-explainer
```
The API's llm_service will now use your fine-tuned, self-hosted model.